# Tutorial de SimulacraBench

Usaremos el esquema de ejemplo
`data/sample.json` para entender la forma de la tarea y la razón más técnica de la competición. Véase `README.md` para el contrato de envío, la regla de puntuación y el reglamento.

In [1]:
import os
import sys
from pathlib import Path

# Este cuaderno vive en tutorials/, pero el arnés vive en la raíz del
# repositorio y todas las rutas de abajo -- config.yml, data/sample.json,
# _sandbox/ -- están escritas respecto a esa raíz. La localizamos y trabajamos
# desde allí, para que el cuaderno funcione igual tanto si Jupyter se inició en
# esta carpeta como en la superior.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("ejecute este cuaderno desde un clon del repositorio")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# Los mismos módulos que usa score.py. Nada de esto reimplementa
# al evaluador: cuando puntúa algo en este cuaderno, está llamando al código
# que le puntúa a usted.
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## 1. La forma de la tarea

`make_sandbox.py` convierte un esquema en un conjunto de datos exactamente con
la forma que tiene el real: mismas columnas, mismas opciones, misma lógica de
salto. Las distribuciones marginales y las dependencias son inventadas, de modo
que una tubería depurada aquí será transferible y un modelo ajustado aquí no lo
será.

Escribe `respondents.parquet` — cada encuestado, más el `role` que dice para
qué sirve — y `schema.json`, que es lo que recibe su `predict()`. Los roles se
deciden una sola vez, aquí, y se escriben en disco. `score.py` los consulta;
nunca parte nada por su cuenta.

In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "visible en ambas fases",
           "DEV": "puntuado en la fase 1, visible en la fase 2",
           "TEST": "puntuado en la fase 2"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"encuestados": counts,
                    "significado": [MEANING[r] for r in counts.index]}).to_string())

A toy instrument, not a real survey. Ten items, few enough to print the whole schema and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. The GIVEN block is deliberately the cheap half of a questionnaire -- the variables that already sit on a sampling frame, a census roster or another survey of the same households -- and the PREDICT block is the expensive half, the part that needs an enumerator and an interview. Use it to see the shape of the task; use the three real schemas to see whether a method works.

       encuestados                                  significado
role                                                           
TRAIN         8000                       visible en ambas fases
TEST          2100                        puntuado en la fase 2
DEV           1900  puntuado en la fase 1, visible en la fase 2


### Qué declara el esquema

Cada ítem lleva cuatro claves: la `question` tal como se formula, una `class`,
los `values` que admite y un `gate` si solo se pregunta a algunas personas.

Las tres clases resumen toda la tarea. **`GIVEN`** es visible para todo el
mundo y nunca se puntúa. **`PREDICT`** se reserva para los encuestados
reservados, y cada celda vacía se puntúa. **`EXCLUDE`** — identificadores,
claves de registro, texto libre — nunca se entrega en la tabla, así que filtre
por `class` en lugar de suponer que el esquema y la tabla llevan las mismas
columnas.

En este instrumento la separación enfrenta deliberadamente la mitad barata de
un cuestionario con la mitad cara. El bloque `GIVEN` es el tipo de variable que
ya figura en un marco muestral, un padrón censal u otra encuesta a los mismos
hogares: dónde vive alguien, cuántos son en el hogar, si hay teléfono. El
bloque `PREDICT` es lo que exige un encuestador y una entrevista. La parte 2 se
apoya en esa distinción.

In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"ítem": name,
                 "clase": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "filtro": gate.get("parent", "-"),
                 "se pregunta si": ", ".join(gate.get("observed_if", [])) or "-",
                 "opciones": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

                  ítem   clase  K         filtro                                        se pregunta si                                                opciones
                region   GIVEN  3              -                                                     -                                 North | Central | South
           urban_rural   GIVEN  2              -                                                     -                                           Urban | Rural
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
        household_size   GIVEN  4              -                                                     -                               1 | 2-3 | 4-5 | 6 or more
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
      has_mobile_phone   GIVEN  2             

`K` es el número de opciones del ítem **incluido el centinela de filtro**: un
ítem filtrado tiene una casilla más que respuestas, porque «nunca preguntado»
es para él una respuesta de pleno derecho. Esa casilla adicional es la última
de su vector de probabilidades, y de `K` se calcula la referencia uniforme `U`.

`would_return` depende de `clinic_wait`, que a su vez depende de
`visited_clinic`: una cadena de profundidad dos. A quien nunca acudió a un
centro de salud nunca se le preguntó cuánto esperó, ni si volvería. Su
respuesta verdadera a ambos es `NA_GATED`.

In [4]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("encuestados").head(12).to_string())

                                                         encuestados
visited_clinic       clinic_wait           would_return             
No                   NA_GATED              NA_GATED             5179
Prefer not to answer NA_GATED              NA_GATED             4413
Yes                  Over 2 hours          Yes                  1405
                     Under 30 minutes      Yes                   380
                     Over 2 hours          No                    204
                                           Not sure              201
                     Under 30 minutes      Not sure              162
                                           No                     28
                     30 minutes to 2 hours Yes                    21
                                           Not sure                4
                                           No                      3


Lea esa tabla como la propia lógica de salto: allí donde `visited_clinic` es
cualquier cosa distinta de `Yes`, ambos ítems hijos son `NA_GATED`, sin
excepción. **La respuesta de un ítem filtrado queda determinada en cuanto su
padre es visible** — eso es puntuación gratis, y lo primero que conviene
explotar.

### Qué recibe `predict()`

`score.py` toma los roles visibles y reservados de la fase, apila a los
encuestados visibles encima de los reservados y vacía cada celda `PREDICT` de
estos últimos. Esas celdas vacías son aquellas para las que usted devuelve
probabilidades.

`NaN` significa exactamente una cosa: *esta celda está reservada, predígala*.
Nunca significa «no respondió» — una no respuesta genuina es una opción
corriente como `Prefer not to answer`, que figura en la lista de opciones como
cualquier otra.

In [5]:
frame, cells, truth = sample_rows(sample, respondents, PHASE)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("tabla:", frame.shape, " celdas a predecir:", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

tabla: (9900, 11)  celdas a predecir: 7600

respondent_id  region urban_rural age_band household_size household_has_children has_mobile_phone visited_clinic  clinic_wait would_return trusts_health_advice
      R000001 Central       Rural      60+            4-5                    Yes               No            Yes Over 2 hours          Yes             Somewhat
      R000002   North       Rural      60+            4-5                    Yes               No             No     NA_GATED     NA_GATED           Not at all
      R000003   South       Rural      60+            2-3                     No               No             No     NA_GATED     NA_GATED                A lot
      R009898   South       Rural      60+            2-3                    Yes               No            NaN          NaN          NaN                  NaN
      R009899   South       Rural      60+            4-5                     No              Yes            NaN          NaN          NaN                  

Las filas de arriba son encuestados visibles: completos, y suyos para aprender
de ellos. Las de abajo están reservadas — usted ve su bloque `GIVEN` y nada
más.

Su valor de retorno es un vector de probabilidades por celda vacía, en **orden
canónico**: las filas de arriba abajo y, dentro de una fila, los ítems en el
orden de las claves de `schema["items"]` — no el orden de `frame.columns`, que
puede diferir. Cada vector sigue los `values` del ítem en orden, más la casilla
centinela cuando el ítem está filtrado. Lea el orden del esquema, nunca de los
datos: una opción que nadie eligió ocupa igualmente una casilla.

In [6]:
print(pd.DataFrame(cells, columns=["fila", "respondent_id", "ítem"]).head(8)
      .to_string(index=False))

 fila respondent_id                 ítem
 8000       R008001       visited_clinic
 8000       R008001          clinic_wait
 8000       R008001         would_return
 8000       R008001 trusts_health_advice
 8001       R008002       visited_clinic
 8001       R008002          clinic_wait
 8001       R008002         would_return
 8001       R008002 trusts_health_advice


### Puntuación

La línea base de la multitud: cada encuestado reservado recibe las
proporciones suavizadas de cada ítem, ignorando todo lo relativo al individuo.
`skill` vale 0 para una respuesta uniforme y 1 para la perfección, y es lo que
ordena la tabla de clasificación.

In [7]:
def hidden_cells(frame, items):
    '''Cada celda vacía, en el orden en que predict() debe devolverlas.'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("referencia uniforme (nats)", "%.4f" % result["uniform_reference"]),
       ("log-score", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("skill vale 0 para una respuesta uniforme y 1 para la perfección.")

referencia uniforme (nats)  1.3144
log-score                   -0.8925
skill                       0.3210

skill vale 0 para una respuesta uniforme y 1 para la perfección.


Ese es todo el contrato. Un envío es un `main.py` con un `predict()` que
devuelve esos vectores; `score.py` lo ejecuta como lo hará el evaluador, y
`tools/check_submission_zip.py` comprueba que el archivo que sube está bien
formado.

---

## 2. Qué aporta un buen modelo

Un benchmark que premia predecir las respuestas de la gente suscita una
inquietud evidente: ¿se trata de dejar de preguntarles? Aquí vemos una forma de combinar predicciones algorítmicas y muestras humanas.

Queremos una sola cifra sobre una población: la proporción de hogares que confían en los consejos de salud de su centro local.
El bloque barato — región, urbano o rural, tamaño del hogar, si hay teléfono —
ya se conoce para cada hogar del marco muestral, a partir de registros
administrativos o de una encuesta anterior. El bloque caro exige un encuestador
en la puerta, y el presupuesto paga unos cientos de entrevistas.

Tiene tres opciones.

1. **Solo entrevistas.** Preguntar a 300 hogares, tomar la proporción y
   publicar un intervalo de confianza. Válido, y tan preciso como permitan 300
   entrevistas.
2. **Solo modelo.** Pasar un modelo por el bloque barato de cada hogar y
   publicar la media. Gratis, y *errado en todo aquello en que el modelo yerre*
   — sin intervalo y sin forma de averiguarlo.
3. **Ambos.** Usar el modelo en todas partes y luego usar las 300 entrevistas
   para medir y restar el error del modelo. Esto es la **inferencia asistida
   por predicción**, y es lo que construye el resto de esta sección.

La tercera es la que merece la pena, porque es válida tanto si el modelo es
bueno como si es malo, y *más precisa que la primera cuando el modelo es
bueno*.

In [8]:
# El estimando: la proporción que confía al menos algo en los consejos de salud.
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# El modelo se ajusta sobre encuestados de rondas anteriores --
# el rol TRAIN, que es exactamente el bloque visible del que aprende un envío.
# Nunca ve los hogares que estamos a punto de entrevistar.
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(bloque barato), para cada hogar

# El marco del que queremos una cifra: los hogares no usados para ajustar el modelo.
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # conocido solo porque los datos son inventados

table([("modelo ajustado sobre, encuestados anteriores:", "%d" % past.sum()),
       ("marco a estimar, hogares:", "%d" % len(frame_rows)),
       ("correlación entre predicción y respuesta:", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("proporción verdadera (que una encuesta real nunca llega a ver):", "%.3f" % TRUTH)])

modelo ajustado sobre, encuestados anteriores:                   8000
marco a estimar, hogares:                                        4000
correlación entre predicción y respuesta:                        0.58
proporción verdadera (que una encuesta real nunca llega a ver):  0.594


Extraigamos ahora las 300 entrevistas y calculemos las tres cifras.

El intervalo de solo entrevistas es el de manual. El intervalo asistido por
predicción es la media del modelo sobre los hogares que **no** entrevistó,
corregida por el error medio del modelo sobre los que sí entrevistó:

```
estimación = media(predicción | no entrevistados) - [ media(predicción | entrevistados) - media(respuesta | entrevistados) ]
                    ↑ el modelo, usado en todas partes      ↑ el error del modelo, medido
```

Ese corchete es todo el mecanismo de seguridad. Se calcula a partir de
respuestas reales, así que cuesta entrevistas reales, y elimina el sesgo del
modelo sea cual sea ese sesgo.

In [9]:
def estimates(f, interviewed, rest):
    '''Estimaciones de solo entrevistas y asistida por predicción, cada una con su error estándar.'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  anchura %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("verdad", "%.3f" % TRUTH),
       ("solo entrevistas", band(classical)),
       ("asistida por predicción", band(powered)),
       ("solo modelo (sin entrevistas)", "%.3f  [sin intervalo alguno]"
        % predicted[frame_rows].mean())])

verdad                         0.594
solo entrevistas               0.597  [0.541, 0.652]  anchura 0.111
asistida por predicción        0.587  [0.541, 0.632]  anchura 0.091
solo modelo (sin entrevistas)  0.590  [sin intervalo alguno]


Una sola extracción no prueba nada — el intervalo pudo tener suerte. Lo que
importa es el comportamiento a lo largo de muchas encuestas: ¿contiene el
intervalo la verdad alrededor del 95 % de las veces, y qué anchura tiene?
Repitamos el ejercicio mil veces, cada una con 300 hogares nuevos.

In [10]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # estimación, error estándar, estimación, error estándar


def summarize(trials, label):
    rows = []
    for name, point, se in (("solo entrevistas", trials[:, 0], trials[:, 1]),
                            ("asistida por predicción", trials[:, 2], trials[:, 3])):
        rows.append({"método": name,
                     "anchura media": (2 * 1.96 * se).mean(),
                     "contiene la verdad": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "un modelo que predice bien")
narrower = 1 - good.loc[1, "anchura media"] / good.loc[0, "anchura media"]
print("\nla banda es un %.0f %% más estrecha, con las mismas %d entrevistas." % (100 * narrower, N_INTERVIEWS))
print("para comprar esa precisión solo con entrevistas harían falta unas %d." % round(N_INTERVIEWS / (1 - narrower) ** 2))

un modelo que predice bien
                 método  anchura media  contiene la verdad
       solo entrevistas          0.111               0.942
asistida por predicción          0.093               0.958

la banda es un 17 % más estrecha, con las mismas 300 entrevistas.
para comprar esa precisión solo con entrevistas harían falta unas 432.


Ambos intervalos contienen la verdad alrededor del 95 % de las veces — eso es
lo que los hace intervalos. El asistido por predicción es simplemente **más
estrecho**, con exactamente el mismo trabajo de campo. Lea la última línea como
la moraleja del ejercicio: un modelo mejor no quita entrevistas del
presupuesto, hace que cada una valga más.

### Qué ocurre cuando el modelo es malo

La objeción evidente es que esto solo funciona mientras el modelo acierte, y
que confiar en él es el riesgo. Aquí está el mismo procedimiento con un modelo
ajustado sobre una **población distinta**.

In [11]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # una población distinta
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("correlación entre predicción y respuesta:", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("solo modelo (sin entrevistas)", "%.3f   frente a una verdad de %.3f   <- desvío de %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "un modelo que no se transfiere")

correlación entre predicción y respuesta:  0.06
solo modelo (sin entrevistas)              0.727   frente a una verdad de 0.594   <- desvío de +0.133



un modelo que no se transfiere
                 método  anchura media  contiene la verdad
       solo entrevistas          0.111               0.942
asistida por predicción          0.117               0.944


,método,anchura media,contiene la verdad
0,solo entrevistas,0.111157,0.942
1,asistida por predicción,0.116888,0.944


Observe: la estimación de solo modelo se equivoca en más de una décima, y nada
en la salida se lo habría dicho — ni intervalo, ni aviso, solo un número que
parece exactamente tan autorizado como el correcto. Sustituir el trabajo de
campo por el modelo introduce sesgo.

El intervalo asistido por predicción sigue conteniendo la verdad alrededor del
95 % de las veces. No es más estrecho que entrevistar a secas — un modelo
inútil no compra precisión. El término de corrección midió el error del modelo
sobre las 300 entrevistas reales y lo restó, que es exactamente para lo que
está.

### Por qué esto necesita un benchmark

La anchura de esa banda es función directa de lo bueno que sea el modelo. De
ahí el interés de medir con cuidado la calidad de predicción, sobre
instrumentos reales, con una regla de puntuación propia: un `skill` mejor en la
clasificación es un intervalo de confianza más estrecho sobre el terreno, o el
mismo intervalo con menos entrevistas.